# Create z.npy from X.npy

This notebook creates `z.npy` which indicates whose turn it is:
- `1` = Player 1's turn
- `-1` = Player 2's turn

The turn is determined by counting the total pieces on the board:
- If total pieces is even → Player 1's turn (they go first)
- If total pieces is odd → Player 2's turn

In [1]:
import numpy as np

In [2]:
# Load the existing data
X = np.load('Dataset/X.npy')
y = np.load('Dataset/y.npy')

print(f"X shape: {X.shape}, dtype: {X.dtype}")
print(f"y shape: {y.shape}, dtype: {y.dtype}")

X shape: (92559, 6, 7, 2), dtype: int8
y shape: (92559,), dtype: int64


In [3]:
# X has shape (N, 6, 7, 2) where:
# - Channel 0: Player 1 pieces (1 where player 1 has a piece)
# - Channel 1: Player 2 pieces (1 where player 2 has a piece)

# Count total pieces for each board
# Sum across all spatial dimensions (6, 7) and both channels (2)
total_pieces = X.sum(axis=(1, 2, 3))

print(f"Total pieces shape: {total_pieces.shape}")
print(f"Total pieces range: [{total_pieces.min()}, {total_pieces.max()}]")
print(f"Sample total pieces (first 10): {total_pieces[:10]}")

Total pieces shape: (92559,)
Total pieces range: [0, 41]
Sample total pieces (first 10): [0 1 2 3 4 5 6 7 8 9]


In [4]:
# Determine whose turn it is:
# - Even total pieces → Player 1's turn (z = 1)
# - Odd total pieces → Player 2's turn (z = -1)

z = np.where(total_pieces % 2 == 0, 1, -1).astype(np.int64)

print(f"z shape: {z.shape}, dtype: {z.dtype}")
print(f"z unique values: {np.unique(z)}")
print(f"z distribution: Player 1's turn: {(z == 1).sum()}, Player 2's turn: {(z == -1).sum()}")

z shape: (92559,), dtype: int64
z unique values: [-1  1]
z distribution: Player 1's turn: 47081, Player 2's turn: 45478


In [5]:
# Save z.npy
np.save('Dataset/z.npy', z)
print("Saved: Dataset/z.npy")

Saved: Dataset/z.npy


In [6]:
# Verify by loading back
z_loaded = np.load('Dataset/z.npy')
print(f"Verification:")
print(f"  z shape: {z_loaded.shape}")
print(f"  z dtype: {z_loaded.dtype}")
print(f"  z unique values: {np.unique(z_loaded)}")
print(f"  Arrays equal: {np.array_equal(z, z_loaded)}")

Verification:
  z shape: (92559,)
  z dtype: int64
  z unique values: [-1  1]
  Arrays equal: True


In [7]:
# Optional: Verify a few samples visually
print("Sample verification:")
for i in range(3):
    board = X[i]
    p1_pieces = board[..., 0].sum()
    p2_pieces = board[..., 1].sum()
    total = p1_pieces + p2_pieces
    turn = z[i]
    print(f"Sample {i}: P1 pieces={p1_pieces}, P2 pieces={p2_pieces}, Total={total}, Turn={turn} ({'P1' if turn == 1 else 'P2'})")

Sample verification:
Sample 0: P1 pieces=0, P2 pieces=0, Total=0, Turn=1 (P1)
Sample 1: P1 pieces=0, P2 pieces=1, Total=1, Turn=-1 (P2)
Sample 2: P1 pieces=1, P2 pieces=1, Total=2, Turn=1 (P1)


# Convert Optimal_Move_Dataset.csv to X4.npy, y4.npy, z4.npy

This section converts the `Optimal_Move_Dataset.csv` to the same format as `X.npy`, `y.npy`, `z.npy`:
- `X4.npy`: Board states with shape (N, 6, 7, 2)
- `y4.npy`: Optimal move (column 0-6)
- `z4.npy`: Whose turn (1 = Player 1, -1 = Player 2)

In [19]:
import pandas as pd

# Load the Optimal_Move_Dataset
df = pd.read_csv('Dataset/Optimal_Move_Dataset.csv')
print(f"Loaded {len(df)} samples")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst 3 rows:")
print(df.head(3))

Loaded 1000000 samples
Columns: ['id', 'player_to_move', 'move_sequence', 'move_count', 'board_flat', 'velena_strong', '3ply', '5ply', '7ply', '9ply']

First 3 rows:
                                 id  player_to_move        move_sequence  \
0  ff120f047f40c4fc94c3b6c601695b77               1     1521542433453163   
1  04f085547655ea01620fef8f1380c484               2  2244350166521531024   
2  6ddb8e236959262d4f2040be7e91e493               1     5015554116035052   

   move_count                                  board_flat  velena_strong  \
0          16  000000000000000002000020112002122100111221              0   
1          19  000000000000000020000022012011212121211121              2   
2          16  000001000000100000020210001012000202122112             -1   

   3ply  5ply  7ply  9ply  
0     0     4     0     6  
1     5     2     3     5  
2     3     2     2     4  


In [20]:
# Filter out rows with invalid moves (velena_strong == -1 means no valid move)
df_valid = df[df['9ply'] >= 0].copy()
print(f"Valid samples (velena_strong >= 0): {len(df_valid)}")
print(f"Removed {len(df) - len(df_valid)} invalid samples")

# Check for any NaN values in board_flat
nan_count = df_valid['board_flat'].isna().sum()
print(f"NaN in board_flat: {nan_count}")
if nan_count > 0:
    df_valid = df_valid.dropna(subset=['board_flat'])
    print(f"After removing NaN: {len(df_valid)} samples")

Valid samples (velena_strong >= 0): 1000000
Removed 0 invalid samples
NaN in board_flat: 0


In [21]:
# Convert board_flat to (N, 6, 7, 2) format
# board_flat is a 42-character string: 0=empty, 1=player1, 2=player2
# The board is stored row by row (row-major order)

N = len(df_valid)
X4 = np.zeros((N, 6, 7, 2), dtype=np.int8)

for idx, (_, row) in enumerate(df_valid.iterrows()):
    flat = str(row['board_flat'])
    for i, char in enumerate(flat):
        r = i // 7  # row
        c = i % 7   # column
        val = int(char)
        if val == 1:
            X4[idx, r, c, 0] = 1  # Player 1
        elif val == 2:
            X4[idx, r, c, 1] = 1  # Player 2
    
    if idx % 200000 == 0:
        print(f"Processed {idx}/{N}...")

print(f"\nX4 shape: {X4.shape}, dtype: {X4.dtype}")

Processed 0/1000000...
Processed 200000/1000000...
Processed 400000/1000000...
Processed 600000/1000000...
Processed 800000/1000000...

X4 shape: (1000000, 6, 7, 2), dtype: int8


In [23]:
# y4: optimal move (velena_strong column, values 0-6)
y4 = df_valid['9ply'].values.astype(np.int64)
print(f"y4 shape: {y4.shape}, dtype: {y4.dtype}")
print(f"y4 unique values: {np.unique(y4)}")
print(f"y4 distribution: {np.bincount(y4)}")

y4 shape: (1000000,), dtype: int64
y4 unique values: [0 1 2 3 4 5 6]
y4 distribution: [108872 130366 155808 208541 156378 130992 109043]


In [24]:
# z4: whose turn (player_to_move column)
# Map: 1 -> 1 (Player 1), 2 -> -1 (Player 2)
player_to_move = df_valid['player_to_move'].values
z4 = np.where(player_to_move == 1, 1, -1).astype(np.int64)
print(f"z4 shape: {z4.shape}, dtype: {z4.dtype}")
print(f"z4 unique values: {np.unique(z4)}")
print(f"z4 distribution: Player 1's turn: {(z4 == 1).sum()}, Player 2's turn: {(z4 == -1).sum()}")

z4 shape: (1000000,), dtype: int64
z4 unique values: [-1  1]
z4 distribution: Player 1's turn: 508903, Player 2's turn: 491097


In [25]:
# Save the arrays
np.save('Dataset/X_9ply.npy', X4)
np.save('Dataset/y_9ply.npy', y4)
np.save('Dataset/z_9ply.npy', z4)

print("Saved:")
print(f"  Dataset/X4.npy: {X4.shape}, dtype: {X4.dtype}")
print(f"  Dataset/y4.npy: {y4.shape}, dtype: {y4.dtype}")
print(f"  Dataset/z4.npy: {z4.shape}, dtype: {z4.dtype}")

Saved:
  Dataset/X4.npy: (1000000, 6, 7, 2), dtype: int8
  Dataset/y4.npy: (1000000,), dtype: int64
  Dataset/z4.npy: (1000000,), dtype: int64


In [26]:
# Verify by loading back
X4_loaded = np.load('Dataset/X3.npy')
y4_loaded = np.load('Dataset/y3.npy')
z4_loaded = np.load('Dataset/z3.npy')
    
print("Verification - loaded shapes:")
print(f"  X4: {X4_loaded.shape}, dtype: {X4_loaded.dtype}")
print(f"  y4: {y4_loaded.shape}, dtype: {y4_loaded.dtype}")
print(f"  z4: {z4_loaded.shape}, dtype: {z4_loaded.dtype}")

# Verify a few samples
print("\nSample verification:")
for i in range(3):
    board = X4_loaded[i]
    p1_pieces = board[..., 0].sum()
    p2_pieces = board[..., 1].sum()
    move = y4_loaded[i]
    turn = z4_loaded[i]
    print(f"Sample {i}: P1={p1_pieces}, P2={p2_pieces}, Move={move}, Turn={turn} ({'P1' if turn == 1 else 'P2'})")

Verification - loaded shapes:
  X4: (857980, 6, 7, 2), dtype: int8
  y4: (857980,), dtype: int64
  z4: (857980,), dtype: int64

Sample verification:
Sample 0: P1=8, P2=8, Move=0, Turn=1 (P1)
Sample 1: P1=10, P2=9, Move=2, Turn=-1 (P2)
Sample 2: P1=11, P2=11, Move=1, Turn=1 (P1)
